In [ ]:
import numpy as np 
from classy import Class 
import matplotlib.pyplot as plt
import pandas as pd
from trianglechain import TriangleChain

In [ ]:
cov = np.loadtxt("/cluster/work/refregier/alexree/local_packages/upanda/src/UPanda/likelihoods/data/desi_bao_2024/cov.txt")
mean = np.loadtxt("/cluster/work/refregier/alexree/local_packages/upanda/src/UPanda/likelihoods/data/desi_bao_2024/mean.txt", usecols=(0, 1))

In [ ]:
def add_om(params_chain):
    param_names = list(params_chain.dtype.names)
    
    oc = params_chain['omega_cdm']
    ob = params_chain['omega_b']
    h = params_chain['h']
    
    try: 
        m_nu = params_chain['m_nu']
    
    except: 
        m_nu = 0.02
        
    o_nu = 3*m_nu/93.14
    
    Om = (oc + ob + o_nu)/(h**2)
    
    #original size 
    n_rows = params_chain.shape
    n_cols = len(param_names)+1
    param_names.append('omega_m')

    
    
    dtype = np.dtype(
        dict(
            formats=[params_chain[str(param_names[0])].dtype] * n_cols,
            names=[f"{param_names[i]}" for i in range(n_cols)],
        )
    )
    rec = np.empty(n_rows, dtype=dtype)
    for i in range(n_cols):
        
        if param_names[i] != 'omega_m':
            rec[param_names[i]] = params_chain[param_names[i]]
        
        else:
            rec[param_names[i]] = Om
            
    return rec

In [ ]:
# Define cosmological parameters
cosmo_params = {
    'h': 0.675,  # Hubble constant divided by 100
    'omega_b': 0.022,  # Baryon density
    'omega_cdm': 0.120,  # Cold dark matter density
    'A_s': 2.1e-9,  # Scalar amplitude
    'n_s': 0.965,  # Scalar spectral index
    'tau_reio': 0.054,  # Reionization optical depth
}

# Initialize CLASS with cosmological parameters
cosmo = Class()
cosmo.set(cosmo_params)
cosmo.compute()

In [ ]:

# Redshifts at which you want predictions
redshifts = [0.295, 0.510, 0.706, 0.930, 1.317, 1.491, 2.330]

# Loop through redshifts to get DM, DH, and DV over rs
for z in redshifts:
    rs = cosmo.rs_drag()  # Sound horizon at drag epoch
    
    # Comoving distance DM
    DM = cosmo.angular_distance(z) * (1 + z)  # In Mpc
    
    # Hubble distance DH
    DH = 1/cosmo.Hubble(z)  # In Mpc

    # Volume distance DV
    DV = (DM**2 * z * DH)**(1/3)  # In Mpc

    # Output predictions normalized by rs
    print(f'z={z:.3f}, DM/rs={DM/rs:.6f}, DH/rs={DH/rs:.6f}, DV/rs={DV/rs:.6f}')
    

### test the implementation

In [ ]:
# load in the DESI 2024 bao emu model 
import UPanda.config as config
from jax import config as jax_config

jax_config.update("jax_enable_x64", True)
config.set_jax_enabled(True)  # noqa: E402
from UPanda import run_panda
from UPanda.integrated_model_jax import IntegratedModel

desi_bao_emu = IntegratedModel(None,None,None,None)
desi_bao_emu.restore("/cluster/work/refregier/alexree/multiprobe_framework/src/multiprobe_framework/combined_probes_data/saved_models/bao")

total_emu_model_list = [[desi_bao_emu]]

In [ ]:
additional_args = {}


In [ ]:
params_chain, loglike =  run_panda.run_mcmc(
    total_emu_model_list = total_emu_model_list,
    experiments_list = ["desi_bao_2024"],
    compression_list = [[], []],
    filename_config="../configs/config_example_desi_bao.yaml",
    n_samples = 5e5,
    n_walkers=128,
    additional_args=additional_args)

In [ ]:
params_chain = add_om(params_chain)

In [ ]:
%matplotlib inline
tri = TriangleChain(density_estimation_method='smoothing', label_fontsize=50)
tri.contour_cl(params_chain, color='steelblue', label = r"DESI BAO", label_fontsize=28, show_values=True)